## Usernames scraping

This code aims to scrap the r/france subreddit to collect all usernames.

### Scrap usernames

In [ ]:
import requests
import time

URL = "https://api.pullpush.io/reddit/submission/search/"
headers = {
    "User-Agent": "Mozilla/5.0 (Our research; contact: placeholder@example.com)"
}

with open("usernames.txt", "r", encoding="utf-8") as f:
    raw_lines = f.readlines()
    lines = []
    for line in raw_lines:
        lines.append(line.strip('\n'))
    last_line = lines[-1]
    try :
        last_line = float(last_line)
        before = last_line
        authors = set(lines[-2::-1])
    except:
        print("No last period saved")
        before = int(time.time())
        authors = set(lines)

unique = len(authors)
while unique < 100000:
    params = {
        "subreddit": "france",
        "size": 100,
        "before": before,
        "sort": "desc"
    }

    r = requests.get(URL,headers=headers, params=params, timeout=30)
    r.raise_for_status()
    data = r.json().get("data", [])

    if not data:
        print("No more data.")
        break

    for post in data:
        author = post.get("author")
        if author and author != "[deleted]":
            authors.add(author)
    
    unique = len(set(authors))

    # prepare for asking previous productions
    before = data[-1]["created_utc"]

    print(f"Authors: {len(authors)} | Unique: {unique}")

    time.sleep(1)


### Store usernames in `usernames.txt`

In [ ]:
number = 0
with open("usernames.txt", "w", encoding="utf-8") as f:
    unique_authors = set(authors)
    for author in unique_authors:
        f.write(f"{author}\n")
    f.write(f'{before}')
